<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/pandas/blob/main/projeto_airbnb_rio/notebooks/airbnb_rio_tratamento_e_analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diagnóstico de preços do Airbnb no Rio de Janeiro

**Demanda:** DADOS-042, Maré Hospedagens (empresa fictícia). Veja o [BRIEFING](../BRIEFING.md).

**Objetivo:** tratar os dados públicos do Inside Airbnb e responder às perguntas de negócio P1 a P6.

**Fonte:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Rio de Janeiro (CC BY 4.0). Data da extração usada: `AAAA-MM-DD` *(preencher)*.

## Sumário
0. Configuração
1. Coleta dos dados
2. Entendimento inicial
3. Tratamento (regras R1 a R12)
4. Outliers
5. Transformações (comodidades, datas, dummies)
6. Análises (P1 a P6)
7. Exportação (CSV e SQLite)
8. Conclusões

## 0. Configuração

In [ ]:
# Importa as bibliotecas usadas no projeto.
# Por quê: pandas/numpy para tratar os dados, matplotlib/seaborn para os gráficos, requests para a API do Banco Central.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Mostra todas as colunas ao exibir um DataFrame (a base tem mais de 70 colunas).
pd.set_option('display.max_columns', None)

In [ ]:
# Log de tratamento (regra R12): guarda quantas linhas existiam antes e depois de cada etapa.
# Por quê: toda remoção de dados precisa ser rastreável no relatório.
log_tratamento = []

def registrar_etapa(etapa, linhas_antes, linhas_depois):
    log_tratamento.append({
        'etapa': etapa,
        'linhas_antes': linhas_antes,
        'linhas_depois': linhas_depois,
        'linhas_removidas': linhas_antes - linhas_depois,
    })

# Exemplo de uso:
# antes = len(df)
# df = df.drop_duplicates()
# registrar_etapa('Remoção de duplicadas', antes, len(df))

## 1. Coleta dos dados

Copie em [insideairbnb.com/get-the-data](https://insideairbnb.com/get-the-data/) os links de `listings.csv.gz` e `calendar.csv.gz` do **Rio de Janeiro**.
O pandas lê `.csv.gz` direto da URL, sem precisar descompactar.

> Revisar no notebook 01: `read_csv` com `usecols` e `nrows`.

In [ ]:
# TODO: colar os links da extração mais recente
URL_LISTINGS = 'COLE_AQUI_O_LINK_DO_listings.csv.gz'
URL_CALENDAR = 'COLE_AQUI_O_LINK_DO_calendar.csv.gz'

In [ ]:
# Colunas usadas na demanda (as outras 60+ não são necessárias).
COLUNAS = [
    'id', 'name', 'host_id', 'host_since', 'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'host_identity_verified', 'host_listings_count',
    'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type',
    'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
    'minimum_nights', 'availability_365', 'number_of_reviews', 'first_review', 'last_review',
    'review_scores_rating', 'review_scores_location', 'review_scores_value', 'instant_bookable',
]

# TODO: ler o listings com read_csv usando usecols=COLUNAS

In [ ]:
# TODO: ler o calendar (é grande: teste antes com nrows=100_000)

## 2. Entendimento inicial

Antes de mexer nos dados: quantas linhas, quais tipos, onde estão os nulos.

> Revisar no notebook 02: `info()`, `isna().sum()`, valores únicos por coluna.

In [ ]:
# TODO: shape, info(), head()

In [ ]:
# TODO: nulos por coluna, em quantidade e em % (isna().sum() / len(df))

In [ ]:
# TODO: valores únicos das colunas categóricas (room_type, host_is_superhost, instant_bookable...)

## 3. Tratamento

Cada etapa registra o antes e o depois com `registrar_etapa()`.

### 3.1 Duplicadas (R5)
> Revisar no notebook 02: `duplicated()` e `drop_duplicates()`.

In [ ]:
# TODO: verificar linhas duplicadas e ids repetidos; remover e registrar no log

### 3.2 Preço (R1 e R6)
O preço vem como texto: `"$1,234.00"`. É preciso remover `$` e `,` e converter para float.

> ⚠️ Cuidado: aqui o `.` é o **separador decimal**, não remova! (é o mesmo erro da célula 72 do notebook 03)

In [ ]:
# TODO: limpar a coluna price com str.replace e converter para float

In [ ]:
# TODO: remover anúncios sem preço (R6) e registrar no log

### 3.3 Percentuais (R2)
`"95%"` vira `0.95`.

In [ ]:
# TODO: tratar host_response_rate e host_acceptance_rate

### 3.4 Colunas t/f (R3)
> Revisar no notebook 02: dicionário de mapeamento + `replace()` (como foi feito com o Churn).

In [ ]:
# TODO: mapear 't' -> 1 e 'f' -> 0 em host_is_superhost, instant_bookable e host_identity_verified

### 3.5 Datas (R4)

In [ ]:
# TODO: converter host_since, first_review e last_review com pd.to_datetime

### 3.6 Banheiros (R10)
`"1.5 baths"` vira `1.5`; `"Half-bath"` vira `0.5`.

> Revisar no notebook 03: regex com `str.extract` / `str.replace`.

In [ ]:
# TODO: criar a coluna numérica 'banheiros' a partir de bathrooms_text

### 3.7 Nulos nas avaliações (R7)
Anúncios sem nenhuma avaliação têm `review_scores_*` nulos. Decida: remover, preencher ou manter, e **justifique** aqui.

In [ ]:
# TODO: tratar os nulos de review_scores_* e registrar no log, se houver remoção

## 4. Outliers de preço (R8)

> Revisar no notebook 02: método IQR (Q1, Q3, IQR, limites), boxplot antes e depois.

In [ ]:
# TODO: boxplot do preço antes do tratamento

In [ ]:
# TODO: calcular Q1, Q3, IQR e os limites; contar quantos anúncios são outliers

In [ ]:
# TODO: aplicar a decisão (remover ou limitar), registrar no log e fazer o boxplot depois

**Decisão sobre os outliers:** *(explicar aqui o que foi feito e por quê)*

## 5. Transformações

### 5.1 Comodidades (R9)
`'["Wifi", "TV", "Kitchen"]'` é uma string, não uma lista.

> Revisar no notebook 03: regex para remover `[ ] "`, `str.split(',')`, `explode()`.

In [ ]:
# TODO: transformar amenities em lista

In [ ]:
# TODO: explode + value_counts para ver as 20 comodidades mais comuns

### 5.2 Tempo como anfitrião (R11)

In [ ]:
# TODO: criar 'anos_como_anfitriao' a partir de host_since (dica: data de hoje - host_since, em anos)

### 5.3 Conferência final do tratamento
Todas as regras do briefing devem estar visíveis aqui.

In [ ]:
# TODO: info(), isna().sum() e o log de tratamento como DataFrame: pd.DataFrame(log_tratamento)

## 6. Análises

> Revisar no notebook 04: `groupby`, `pivot_table`, `idxmax`, gráficos com `.plot()`.

### P1. Os 10 bairros mais caros e os 10 mais baratos
Use a **mediana** do preço (mais robusta que a média) e considere só bairros com um número mínimo de anúncios.

In [ ]:
# TODO

### P2. Preço por tipo de quarto em cada bairro

In [ ]:
# TODO: pivot_table com index=bairro, columns=room_type, values=price, aggfunc='median'

### P3. Superhosts cobram mais e têm avaliações melhores?

In [ ]:
# TODO

### P4. Comodidades ligadas a preços maiores
Ideia: para cada comodidade de interesse (piscina, vista para o mar, ar-condicionado...), compare o preço mediano de quem tem e de quem não tem.

In [ ]:
# TODO

### P5. Sazonalidade: preço e ocupação por mês (Carnaval e Réveillon)
Use o `calendar`. Se a coluna de preço vier vazia nessa extração, analise só a disponibilidade (`available`).

> Revisar no notebook 03: `to_datetime`, `dt.strftime('%Y-%m')` e `groupby`.

In [ ]:
# TODO

### P6. Diária mediana em dólar
Cotação do dólar pela API do Banco Central (série 1, dólar comercial).

> Revisar no notebook 01: `requests.get()`, `.json()` e `json_normalize`.

In [ ]:
URL_DOLAR = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados/ultimos/1?formato=json'

# TODO: buscar a cotação, converter o valor para float e calcular a diária mediana em US$

## 7. Exportação

> Revisar no notebook 01: `to_csv` e SQLAlchemy (`create_engine`, `to_sql`, `read_sql`).

In [ ]:
# TODO: salvar em ../dados/tratados/listings_tratado.csv (sem o índice)

In [ ]:
# TODO: salvar no SQLite ../dados/tratados/airbnb_rio.db, tabela 'listings'

In [ ]:
# TODO: 3 consultas SQL de exemplo com pd.read_sql

## 8. Conclusões

| Pergunta | Resposta |
| --- | --- |
| P1 | |
| P2 | |
| P3 | |
| P4 | |
| P5 | |
| P6 | |

**Limitações:** *(ex.: preço do anúncio não é o preço pago; dados de uma única extração...)*

**Próximos passos:** *(ex.: modelo de previsão de preço, comparar com outras cidades...)*